# NB00 — Plan del TFM

**TFM Víctor Rodríguez · UIV 2025-2026 · Dir. Karen López-Linares**

---

## ¿Qué problema resolvemos?

Construir un **clasificador binario de mamografía de cribado** que decida si un estudio (o una mama individual) presenta lesiones sospechosas, partiendo de los pesos preentrenados de **AsymMirai** y aprendiendo solo una cabeza propia encima.

## ¿Por qué AsymMirai y no entrenar todo desde cero?

AsymMirai se compone de dos partes preentrenadas con orígenes distintos:

1. Un **backbone ResNet-18** heredado de **Mirai** (Yala et al., 2021), entrenado en el Massachusetts General Hospital sobre aproximadamente **56.000 mamografías** con seguimiento longitudinal. Aprendió representaciones del dominio: texturas mamográficas, tejido fibroglandular, calcificaciones.
2. Una **lógica de asimetría bilateral** propia de AsymMirai (pesos de stretch aprendidos, normalización del score), entrenada por Donnelly et al. (2024) sobre el dataset **EMBED** (Emory BrEast imaging Dataset, 70.136 pacientes en la cohorte de training). La intuición clínica es que asimetrías bilaterales son señal de malignidad en muchos casos.

Su salida original (un score de riesgo a 1-5 años) **no es útil para nosotros** porque VinDr-Mammo y RSNA no tienen seguimiento longitudinal — solo tenemos BI-RADS del examen actual. Pero los **pesos** sí son útiles: encapsulan conocimiento aprendido con muchos más datos de los que nosotros podríamos reunir.

**Estrategia:** congelamos AsymMirai, extraemos sus representaciones internas, y entrenamos una cabeza pequeña encima.

## ¿Qué tenemos para extraer?

Mirando el `forward` de `LocalizedDifModel` (la clase principal de AsymMirai):

```python
for (left, right, view) in [(L_CC, R_CC, 'CC'), (L_MLO, R_MLO, 'MLO')]:
    left_emb  = self.backbone(left)             # ──► PUNTO A
    right_emb = self.backbone(right)            # ──► PUNTO A
    asym, _   = self.asymmetry_metric(L, R, ...)  # ──► PUNTO B = |L_stretched − R_stretched|
```

Hay dos puntos naturales para cortar la red y reutilizar lo aprendido:

| Punto | Qué es | Qué aprovechamos de AsymMirai |
|-------|--------|-------------------------------|
| **A** | Embeddings del backbone, uno por vista | Solo el ResNet-18 preentrenado, sin la innovación bilateral |
| **B** | Diferencia bilateral abs `|L − R|` con pesos de stretch aprendidos | Backbone + **la innovación específica** de AsymMirai (asimetría) |

**Pregunta principal del TFM:** ¿la innovación de AsymMirai (asimetría bilateral) aporta valor sobre solo usar su backbone? Si Punto B > Punto A, hemos validado el sesgo inductivo de AsymMirai en una tarea distinta a la original.

## Reducción de dimensión: GAP + GMP

Los mapas del backbone son `(512, 52, 64)` por vista = 1,7 millones de números. Inviable guardar los 5.000 estudios sin reducir. Aplicamos **dos poolings combinados**:

- **GAP** (Global Average Pooling): promedio espacial de cada canal → 512 dims. Información general.
- **GMP** (Global Max Pooling): pico máximo de cada canal → 512 dims. Información local concentrada en pequeñas regiones (justo donde aparecen lesiones).

Concatenados: **1024 dims por vista o por par bilateral**. Aplicamos la misma reducción a Punto A y Punto B para que la comparación entre ambos sea limpia (mismo método de pooling, solo cambia el contenido).

## Granularidad: a qué nivel clasificamos

Hacemos dos experimentos en paralelo porque cada uno responde una pregunta clínica distinta:

### Nivel ESTUDIO (n = 5.000, prevalencia ~10%)
Etiqueta = `max BI-RADS de las 4 vistas ≥ 4`. Decisión clínica: "este examen requiere segunda opinión".
Aquí podemos usar **ambos puntos**:
- **E-A**: concat de las 4 vistas en Punto A → **4096 dims**.
- **E-B**: concat de los 2 pares bilaterales en Punto B → **2048 dims**.

### Nivel MAMA (n = 10.000, prevalencia ~5%)
Etiqueta = `max(BI-RADS_CC, BI-RADS_MLO) ≥ 4`. Decisión clínica: "¿qué mama es la sospechosa?".
Aquí **solo aplica el Punto A** porque la asimetría compara las dos mamas entre sí — no tiene sentido evaluar la asimetría "de una mama sola".
- **M-A**: concat(CC, MLO) de esa mama → **2048 dims**.

## Clasificadores que comparamos

Sobre cada configuración entrenamos **dos cabezas distintas** con el mismo tratamiento de desbalance:

- **MLP** (PyTorch): capa oculta con BatchNorm + Dropout, BCE con `pos_weight`. Aprende combinaciones no lineales.
- **GBM** (LightGBM): gradient boosting de árboles con `scale_pos_weight`. Aprende reglas de decisión sobre features individuales.

Comparar MLP vs GBM sobre las mismas features responde a la pregunta: ¿qué tipo de cabeza explota mejor las representaciones de AsymMirai?

## Resultado: 6 configuraciones a entrenar

| Nombre | Granularidad | Punto | Cabeza | Dims | n | prev |
|--------|--------------|-------|--------|------|-----------|------|
| E_A_mlp | Estudio | A | MLP | 4096 | 5.000 | ~10% |
| E_A_gbm | Estudio | A | LightGBM | 4096 | 5.000 | ~10% |
| E_B_mlp | Estudio | B | MLP | 2048 | 5.000 | ~10% |
| E_B_gbm | Estudio | B | LightGBM | 2048 | 5.000 | ~10% |
| M_A_mlp | Mama | A | MLP | 2048 | 10.000 | ~5% |
| M_A_gbm | Mama | A | LightGBM | 2048 | 10.000 | ~5% |

## Etiqueta binaria

VinDr-Mammo asigna BI-RADS 1-5 a cada mama. Mapeo a binario según relevancia clínica:

| BI-RADS | Etiqueta | Justificación |
|---------|----------|---------------|
| 1, 2, 3 | 0 (no sospechoso) | 1=negativo, 2=benigno, 3=probablemente benigno con seguimiento. No requieren derivación inmediata. |
| 4, 5    | 1 (sospechoso)    | Sospechoso o altamente sospechoso de malignidad. Deriva a biopsia. |

## Validación y split

Respetamos el split predefinido de VinDr (`training` 4.000 / `test` 1.000). Sobre `training` aplicamos **StratifiedKFold(5)** para obtener predicciones out-of-fold. El `test` queda intacto como hold-out limpio y se evalúa solo al final con el ensemble de los 5 modelos.

A nivel mama usamos **StratifiedGroupKFold** con `groups=study_id` para evitar que las dos mamas del mismo paciente caigan en folds distintos (eso inflaría el AUC artificialmente).

## Estructura de notebooks

```
00_overview.ipynb              ← este notebook
01_inspeccion_modelo.ipynb     ← cargar AsymMirai y verificar atributos
02_extraccion_un_estudio.ipynb ← validar pipeline GAP+GMP sobre 1 estudio
03_extraccion_masiva.ipynb     ← extraer features de los 5.000 estudios
04_definicion_modelos.ipynb    ← cabezas MLP y wrappers LightGBM
05_entrenamiento_kfold.ipynb   ← entrenar las 6 configuraciones
06_evaluacion.ipynb            ← AUC, ROC, DeLong, IC bootstrap
07_fusion_densidad.ipynb       ← añadir covariable clínica sobre la mejor config
```

## Verificación de rutas y configuración compartida

La celda siguiente crea las carpetas de salida y comprueba que los archivos críticos están donde deben.

In [1]:
import os
from pathlib import Path

# Raíz del proyecto. Cambiar solo aquí si el proyecto se mueve.
# Raíz del proyecto: por defecto, la carpeta padre de notebooks/.
# Sobrescribible con la variable de entorno TFM_PROJECT_ROOT.
BASE      = os.environ.get('TFM_PROJECT_ROOT',
                           os.path.abspath(os.path.join(os.getcwd(), '..')))

# Subcarpetas
ASYMMIRAI    = os.path.join(BASE, 'AsymMirai')
DATA         = os.path.join(BASE, 'Data', 'vindr-mammo')
NOTEBOOKS    = os.path.join(BASE, 'Notebooks')
OUTPUTS      = os.path.join(BASE, 'outputs')
FEATURES_DIR = os.path.join(OUTPUTS, 'Features')
PRED_DIR     = os.path.join(OUTPUTS, 'Predicciones')
MODELS_DIR   = os.path.join(OUTPUTS, 'Models')
PLOTS_DIR    = os.path.join(OUTPUTS, 'Plots')

for d in (FEATURES_DIR, PRED_DIR, MODELS_DIR, PLOTS_DIR):
    Path(d).mkdir(parents=True, exist_ok=True)

# Archivos clave
WEIGHTS    = os.path.join(ASYMMIRAI, 'snapshots', 'trained_asymmirai.pt')
BREAST_CSV = os.path.join(DATA, 'breast-level_annotations.csv')
IMG_DIR    = os.path.join(DATA, 'images')

SEED = 42

print('Sanity check de rutas:')
for name, p in [('Pesos AsymMirai',   WEIGHTS),
                ('Annotations VinDr', BREAST_CSV),
                ('Carpeta imágenes',  IMG_DIR),
                ('Notebooks',         NOTEBOOKS)]:
    sym = '✓' if os.path.exists(p) else '✗'
    print(f'  {sym}  {name}: {p}')

print('\nCarpetas de salida (creadas si no existían):')
for name, p in [('Features', FEATURES_DIR), ('Predicciones', PRED_DIR),
                ('Models', MODELS_DIR), ('Plots', PLOTS_DIR)]:
    print(f'  ✓ {name}: {p}')

print('\nListo. Si todo está ✓, pasa al NB01 (o directo al NB02 si NB01 ya está ejecutado).')

Sanity check de rutas:
  ✓  Pesos AsymMirai: C:\Users\victo\Documents\TFM\Proyecto\AsymMirai\snapshots\trained_asymmirai.pt
  ✓  Annotations VinDr: C:\Users\victo\Documents\TFM\Proyecto\Data\vindr-mammo\breast-level_annotations.csv
  ✓  Carpeta imágenes: C:\Users\victo\Documents\TFM\Proyecto\Data\vindr-mammo\images
  ✓  Notebooks: C:\Users\victo\Documents\TFM\Proyecto\Notebooks

Carpetas de salida (creadas si no existían):
  ✓ Features: C:\Users\victo\Documents\TFM\Proyecto\Outputs\Features
  ✓ Predicciones: C:\Users\victo\Documents\TFM\Proyecto\Outputs\Predicciones
  ✓ Models: C:\Users\victo\Documents\TFM\Proyecto\Outputs\Models
  ✓ Plots: C:\Users\victo\Documents\TFM\Proyecto\Outputs\Plots

Listo. Si todo está ✓, pasa al NB01 (o directo al NB02 si NB01 ya está ejecutado).
